In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st
import urllib
import matplotlib.image as mpimg
from babel.numbers import format_currency

sns.set(style='whitegrid')

# --- HELPER FUNCTIONS ---

def create_category_revenue_df(df):
    category_rev = df.groupby("product_category_name_english").agg({
        "order_id": "nunique",
        "price": "sum"
    }).sort_values(by="price", ascending=False).reset_index()
    category_rev.rename(columns={"order_id": "total_orders", "price": "total_revenue"}, inplace=True)
    return category_rev

def create_state_revenue_df(df):
    state_rev = df.groupby("customer_state").agg({
        "customer_unique_id": "nunique",
        "payment_value": "sum"
    }).sort_values(by="payment_value", ascending=False).reset_index()
    return state_rev

def create_rfm_segment_df(df):
    customer_spend = df.groupby('customer_unique_id')['payment_value'].sum().reset_index()
    def segmenting(total_spend):
        if total_spend > 500: return 'High Spender'
        elif total_spend > 150: return 'Medium Spender'
        else: return 'Low Spender'
    customer_spend['segment'] = customer_spend['payment_value'].apply(segmenting)
    return customer_spend

# --- LOAD DATA ---
# Di dalam dashboard.py
import pandas as pd
import streamlit as st

# Membaca data langsung dari file zip
# Pastikan nama file di dalam zip adalah 'main_data.csv'
try:
    all_df = pd.read_csv("main_data.zip")
except Exception as e:
    # Jika gagal membaca zip, coba baca csv biasa (sebagai cadangan)
    all_df = pd.read_csv("main_data.csv")

# --- SIDEBAR FILTER ---
min_date = all_df["order_purchase_timestamp"].min()
max_date = all_df["order_purchase_timestamp"].max()

with st.sidebar:
    st.image("https://github.com/dicodingacademy/assets/raw/main/logo.png")
    start_date, end_date = st.date_input(
        label='Rentang Waktu Analysis',
        min_value=min_date,
        max_value=max_date,
        value=[min_date, max_date]
    )

main_df = all_df[(all_df["order_purchase_timestamp"] >= str(start_date)) &
                (all_df["order_purchase_timestamp"] <= str(end_date))]

# --- PREPARE DATAFRAMES ---
category_revenue_df = create_category_revenue_df(main_df)
state_revenue_df = create_state_revenue_df(main_df)
rfm_segment_df = create_rfm_segment_df(main_df)

# --- DASHBOARD HEADER ---
st.header('Mareska E-Commerce Analysis Dashboard :sparkles:')

# --- PERTANYAAN 1: PRODUCT PERFORMANCE ---
st.subheader("Product Performance: Top & Bottom Category by Revenue")
col1, col2 = st.columns(2)

with col1:
    total_rev = format_currency(category_revenue_df.total_revenue.sum(), "BRL", locale='es_CO')
    st.metric("Total Revenue", value=total_rev)

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(24, 10))
colors = ["#72BCD4", "#D3D3D3", "#D3D3D3", "#D3D3D3", "#D3D3D3"]

sns.barplot(x="total_revenue", y="product_category_name_english", data=category_revenue_df.head(5), palette=colors, ax=ax[0])
ax[0].set_title("Top 5 Product Categories by Revenue", loc="center", fontsize=25)
ax[0].tick_params(axis='y', labelsize=20)

sns.barplot(x="total_revenue", y="product_category_name_english", data=category_revenue_df.sort_values(by="total_revenue", ascending=True).head(5), palette=colors, ax=ax[1])
ax[1].set_title("Bottom 5 Product Categories by Revenue", loc="center", fontsize=25)
ax[1].invert_xaxis()
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
ax[1].tick_params(axis='y', labelsize=20)

st.pyplot(fig)

# --- PERTANYAAN 2: GEOSPATIAL REVENUE ---
st.subheader("Geospatial Revenue: Total Transaction by States")
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x="payment_value", y="customer_state", data=state_revenue_df.head(10), palette="viridis", ax=ax)
ax.set_title("Top 10 States by Total Transaction Value", fontsize=15)
st.pyplot(fig)

# --- PERTANYAAN 3: GEOSPATIAL DISTRIBUTION (MAP) ---
st.subheader("Customer Geospatial Distribution")

def plot_brazil_map(df):
    # Pastikan kolom yang dibutuhkan ada
    if 'geolocation_lng' in df.columns and 'geolocation_lat' in df.columns:
        brazil_img = mpimg.imread(urllib.request.urlopen('https://i.pinimg.com/originals/3a/0c/e1/3a0ce18b3c842748c255bc0aa445ad41.jpg'),'jpg')
        fig, ax = plt.subplots(figsize=(10, 10))

        # Menggunakan data yang unik per pelanggan agar tidak berat
        df_plot = df.drop_duplicates(subset='customer_unique_id')

        ax.scatter(df_plot["geolocation_lng"], df_plot["geolocation_lat"], alpha=0.3, s=0.3, c='maroon')

        plt.imshow(brazil_img, extent=[-73.98, -33.8, -33.75, 5.4])
        plt.title('Distribusi Geografis Pelanggan di Brazil')
        plt.axis('off')
        st.pyplot(fig)
    else:
        st.warning("Data koordinat geolokasi tidak ditemukan untuk visualisasi peta.")

# Panggil fungsi
plot_brazil_map(main_df)

# --- ANALISIS LANJUTAN: SEGMENTASI ---
st.subheader("Customer Value Segmentation")
fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(x='segment', data=rfm_segment_df, palette='viridis', order=['Low Spender', 'Medium Spender', 'High Spender'], ax=ax)
st.pyplot(fig)

st.caption('Copyright (c) Mareska Radela Putra 2026')


2026-03-24 12:27:11.337 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 12:27:11.339 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 12:27:11.340 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 12:27:11.340 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 12:27:11.341 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 12:27:11.342 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 12:27:11.343 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 12:27:11.344 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()